# Custom Energy Parametrization
Demonstrate MeshFEM's custom energy definition.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
# os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import MeshFEM
import mesh, py_newton_optimizer, viewer
import param_utils, benchmark
import numpy as np

m = mesh.Mesh('../models/hand.msh')

In [ ]:
import continuation_parametrization
from continuation_parametrization import parametrization, energy

In [ ]:
# Scale so that the surface area is pi (to match [Su et al. 2020])
m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))

In [ ]:
import mesh_energy
uv = mesh_energy.NodalVars(m, 2)
uv.setVars(param_utils.tutteInitialization(m).ravel())

In [ ]:
param = parametrization.Parametrization(m, uv, energy.SymmetricARAP(2))
objectives = [param]

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)

In [ ]:
# Only run this cell if you're using a locally injective energy!
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.9

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
prob.hessianShift = 1e-8 # Work around energy nullspace by adding a small shift
prob.useRelativeHessianShift = True
opt = prob.optimizer()
opt.options.niter = 500
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
opt.options.hessianProjectionController.startWithProjectionActive = True
# prob.setCustomIterationCallback(v.updater())

In [ ]:
benchmark.reset()
rep = opt.optimize()
benchmark.report()
v.update()